<img src="../assets/agent_design_lens.png" alt="Agent 설계 요소 개념도" style="display: block; width: 100%; max-width: 1200px; margin: 18px auto 24px; border-radius: 12px;">


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">1. Prompt Chatbot (UI)</h2>
<p style="margin: 7px 0;">아래 UI에서 system prompt, 모델, 실행 방식을 바꿔 보세요. ReAct Agent는 계산이 필요할 때 도구를 호출해 결과를 답변에 반영합니다.</p>
<ol style="margin: 7px 0 0; padding-left: 1.3em;">
<li>질문을 입력해 답변을 확인합니다.</li>
<li>⚙️ 설정에서 일반 Chat 또는 ReAct Agent를 선택합니다. ReAct Agent에는 <code>15% 할인 뒤 3개 가격은?</code>처럼 계산 질문을 입력해 보세요.</li>
</ol>
  </div>
</div>


In [1]:
from pathlib import Path
import json
from time import perf_counter
import sys
project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / '1_gen_ai_hub' / 'src').exists())
sys.path[:0] = [str(project_root / '1_gen_ai_hub' / 'src'), str(project_root / '2_gen_ai_hub_llm')]
from src.prompt_model.lab_core import run_adaptive_react_agent, run_chat, run_chat_stream, run_react_agent
from src.notebook_ui import build_chat_ui
chat_ui = build_chat_ui(run_chat, chat_stream_runner=run_chat_stream, react_agent_runner=run_react_agent, adaptive_react_agent_runner=run_adaptive_react_agent)
chat_ui.launch(inline=True, share=False, server_name='127.0.0.1', show_error=True, width='100%', height=720)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1a73e8; background: #e8f0fe; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1a73e8; letter-spacing: .02em;">✅  체크포인트</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">메시지 role과 준비</h2>
<ul style="margin: 7px 0 0; padding-left: 1.3em;">
<li><code>system</code>은 모델의 역할, 규칙, 언어, 출력 형식을 정합니다.</li>
<li><code>user</code>는 사용자의 질문 또는 요청입니다.</li>
<li><code>assistant</code>는 이전 모델 응답으로, 대화 history에 사용합니다.</li>
</ul>
<p style="margin: 7px 0;">아래 코드는 루트 <code>.env</code>의 <code>LLM_MODEL</code> 설정을 사용합니다.</p>
  </div>
</div>


<hr style="border: 0; border-top: 2px solid #c7d2e1; margin: 32px 0 14px;">
<div style="margin: 0 0 18px; color: #4b5f78; font-size: .86em; font-weight: 700; letter-spacing: .04em;">🧩 코드로 확인하기</div>


In [2]:
from pathlib import Path
import sys
project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / '1_gen_ai_hub' / 'src').exists())
sys.path[:0] = [str(project_root / '1_gen_ai_hub' / 'src'), str(project_root / '2_gen_ai_hub_llm')]
from src.prompt_model.lesson_support.llm import chat_text
from src.prompt_model.lesson_support.config import load_settings
from src.prompt_model.lab_core import compare_prompts, run_chat
settings = load_settings()
print(f'LLM model: {settings.llm_model}')

LLM model: gpt-5.6-luna


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #5f6368; background: #f1f3f4; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #5f6368; letter-spacing: .02em;">🔎  선택 심화</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">선택: 코드로 prompt 비교하기</h2>
<p style="margin: 7px 0;">UI에서 본 차이를 <code>compare_prompts()</code>로 확인하는 선택 실습입니다.</p>
<h3 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">기본 chat 응답</h3>
<p style="margin: 7px 0;">같은 질문을 기본 조건과 표 형식 조건으로 비교합니다.</p>
  </div>
</div>


In [3]:
question = 'LLM의 기본 아키텍처를 설명해줘.'

comparison_system_prompt = """너는 친절한 선생님이다.
초등학생도 이해할 수 있게 쉬운 한국어로 설명한다.
첫 줄에는 한 문장 요약을 쓴다.
그 아래에 핵심 내용을 bullet 3개로 정리한다."""

# comparison_system_prompt를 직접 바꿔 원하는 제약을 실험하세요.
comparison = compare_prompts(question, settings, system_prompt=comparison_system_prompt)
print(comparison['baseline'])

LLM(Large Language Model)의 대표적인 기본 아키텍처는 **Transformer**입니다. 특히 ChatGPT와 같은 생성형 LLM은 주로 **Decoder-only Transformer** 구조를 사용합니다.

## 1. 입력 토큰화(Tokenization)

문장을 모델이 처리할 수 있도록 작은 단위인 **토큰(token)**으로 나눕니다.

예를 들어:

> “나는 인공지능을 공부한다”

가 다음과 같은 토큰 ID로 변환될 수 있습니다.

```text
[나는] [인공지능] [을] [공부] [한다]
→ [1023, 4581, 27, 912, 334]
```

토큰은 단어 전체일 수도 있고, 단어의 일부나 문자일 수도 있습니다.

---

## 2. 임베딩(Embedding)

각 토큰 ID를 고정된 길이의 숫자 벡터로 변환합니다.

```text
토큰 ID → 벡터
1023 → [0.12, -0.34, 0.87, ...]
```

이 벡터는 토큰의 의미와 사용 패턴을 학습한 표현입니다.

Transformer는 토큰의 순서 정보를 별도로 알아야 하므로 **위치 정보(Positional Information)**도 추가합니다.

```text
입력 표현 = 토큰 임베딩 + 위치 임베딩
```

최근 모델에서는 전통적인 위치 임베딩 대신 RoPE(Rotary Positional Embedding) 같은 방식을 자주 사용합니다.

---

## 3. Transformer 블록

LLM은 수십 개에서 수백 개의 Transformer 블록을 쌓은 형태입니다. 각 블록은 보통 다음 요소로 구성됩니다.

```text
입력
 ↓
Self-Attention
 ↓
Feed-Forward Network
 ↓
출력
```

### 3.1 Self-Attention

Self-Attention은 문장 안의 각 토큰이 다른 토큰들을 얼마나 참고해야 하는지 계산합니다.

예를 들어:

> “철수가 사과를 먹었다. 그는 맛있다고 말했다.”

여기서 “그”가 “철수

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h3 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">system prompt로 제약하기</h3>
<p style="margin: 7px 0;">제목, 표, bullet처럼 확인하기 쉬운 형식을 지정했을 때 응답이 어떻게 달라지는지 봅니다.</p>
  </div>
</div>


In [4]:
print(comparison['constrained'])

LLM은 글을 숫자로 바꾸고, 문맥을 살펴 다음에 올 말을 예측하는 인공지능입니다.

- **입력 변환**: 문장을 작은 단위인 **토큰**으로 나누고, 각 토큰을 숫자 벡터인 **임베딩**으로 바꿉니다. 단어의 순서를 알 수 있도록 위치 정보도 더합니다.
- **문맥 이해**: 여러 개의 **트랜스포머 층**이 차례로 정보를 처리합니다. 그 중심 기능인 **어텐션**은 문장 속에서 서로 관련 있는 단어를 찾아 “어떤 단어를 중요하게 볼지” 결정합니다. 그 뒤 신경망이 정보를 한 번 더 정리합니다.
- **다음 단어 예측**: 마지막 층은 가능한 다음 토큰마다 점수를 계산하고, **소프트맥스**로 확률을 만듭니다. LLM은 가장 알맞은 토큰을 하나씩 골라 문장을 계속 만들어 갑니다.


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  실습 2</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">message 배열을 직접 구성하기</h2>
<p style="margin: 7px 0;">LLM API에는 역할이 붙은 메시지 배열을 보냅니다. 아래에서 <code>system</code> 규칙, 이전 대화, 새 질문의 순서를 확인해 보세요.</p>
  </div>
</div>


In [5]:
messages = [
    {"role": "system", "content": "너는 친절한 교육 보조자다. 한국어로 간결하게 답한다."},
    {"role": "user", "content": "LLM의 장점을 두 가지 알려줘."},
    {"role": "assistant", "content": "자연어를 이해하고 다양한 텍스트 작업을 도울 수 있습니다."},
    {"role": "user", "content": "방금 답을 표로 정리해줘."},
]
for index, item in enumerate(messages, start=1):
    print(f'{index}. {item["role"]}: {item["content"]}')

reply = chat_text(settings, messages, settings.llm_model)
print()
print('응답:')
print(reply)

1. system: 너는 친절한 교육 보조자다. 한국어로 간결하게 답한다.
2. user: LLM의 장점을 두 가지 알려줘.
3. assistant: 자연어를 이해하고 다양한 텍스트 작업을 도울 수 있습니다.
4. user: 방금 답을 표로 정리해줘.

응답:
| 장점 | 설명 |
|---|---|
| 자연어 이해 | 사람의 질문과 문장을 이해하고 자연스럽게 답변할 수 있습니다. |
| 다양한 작업 지원 | 글쓰기, 요약, 번역, 아이디어 생성 등 여러 작업을 도울 수 있습니다. |


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  실습 3</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">대화 history가 맥락을 전달하는 방식</h2>
<p style="margin: 7px 0;">두 번째 질문만으로는 무엇을 표로 바꿔야 하는지 알기 어렵습니다. history 유무에 따른 결과를 비교해 보세요.</p>
<p style="margin: 7px 0;">최근 12개 메시지, 총 12,000자까지만 history를 유지합니다.</p>
  </div>
</div>


In [6]:
follow_up = '방금 답을 표로 정리해줘.'
history = [
    {"role": "user", "content": "LLM의 장점을 두 가지 알려줘."},
    {"role": "assistant", "content": "자연어를 이해하고 다양한 텍스트 작업을 도울 수 있습니다."},
]
without_history = run_chat(follow_up, settings)
with_history = run_chat(follow_up, settings, history=history)

print('='*20,'[history 없음]', '='*20)
print(without_history['reply'])
print()
print('='*20, '[history 포함]','='*20)
print(with_history['reply'])

==================== [history 없음] ====================
정리할 **이전 답변 내용이 현재 대화에 보이지 않습니다.**  
표로 만들 원문이나 내용을 보내주시면 아래와 같이 정리해드릴게요.

| 항목 | 내용 |
|---|---|
| 핵심 내용 1 |  |
| 핵심 내용 2 |  |
| 장점 |  |
| 주의사항 |  |
| 결론 |  |

==================== [history 포함] ====================
| 장점 | 설명 |
|---|---|
| 자연어 이해 | 사람의 질문과 문장을 이해하고 자연스럽게 답변할 수 있습니다. |
| 다양한 텍스트 작업 지원 | 요약, 번역, 글쓰기, 문서 작성 등 여러 작업을 도울 수 있습니다. |


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  실습 4</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">프롬프트로 JSON 형식 요청하고 검증하기</h2>
<p style="margin: 7px 0;">비정형 고객 리뷰에서 긍정·부정 의견, 요약, 키워드를 JSON으로 추출한 뒤 <code>json.loads()</code>로 실제 파싱을 검증합니다.</p>
  </div>
</div>


In [7]:
# 1) 고객이 남긴 비정형 리뷰 원문입니다.
customer_review = (
    '배송은 하루 만에 와서 정말 빨랐고 포장도 깔끔했어요. 제품 디자인과 화면은 마음에 듭니다. ' 
    '다만 배터리가 생각보다 빨리 닳고, 앱 연결이 가끔 끊겨서 아쉬워요. ' 
    '다음 업데이트에서 연결 안정성이 좋아지면 친구에게도 추천할 것 같아요.'
)
print('[1. 비정형 고객 리뷰]')
print(customer_review)

# 2) 리뷰에서 긍정·부정 의견과 핵심 정보를 JSON으로 추출합니다.
json_format_prompt = (
    '너는 고객 리뷰를 분석하는 교육 보조자다. 반드시 JSON 객체만 반환한다. ' 
    'Markdown 코드 블록이나 설명을 덧붙이지 않는다. ' 
    '키는 positive(문자열 배열), negative(문자열 배열), summary(문자열), keywords(문자열 배열)만 사용한다.'
)
json_outcome = run_chat(
    f'다음 고객 리뷰를 지정한 JSON 형식으로 분석해줘.\n\n{customer_review}',
    settings,
    system_prompt=json_format_prompt,
)
json_text = json_outcome['reply']
print('\n[2. 리뷰 분석 JSON]')
print(json_text)

# 3) 모델의 반환값을 실제 JSON으로 파싱해 형식을 검증합니다.
try:
    parsed = json.loads(json_text)
    print('\n[3. JSON 파싱 성공]')
    print(parsed)
except json.JSONDecodeError:
    print('\n[3. JSON 파싱 실패] 반환 원문을 확인하세요:')
    print(json_text)

[1. 비정형 고객 리뷰]
배송은 하루 만에 와서 정말 빨랐고 포장도 깔끔했어요. 제품 디자인과 화면은 마음에 듭니다. 다만 배터리가 생각보다 빨리 닳고, 앱 연결이 가끔 끊겨서 아쉬워요. 다음 업데이트에서 연결 안정성이 좋아지면 친구에게도 추천할 것 같아요.

[2. 리뷰 분석 JSON]
{"positive":["배송이 하루 만에 도착해 매우 빨랐음","포장이 깔끔함","제품 디자인과 화면이 마음에 듦"],"negative":["배터리가 생각보다 빨리 닳음","앱 연결이 가끔 끊김","연결 안정성 개선이 필요함"],"summary":"배송과 포장, 디자인과 화면에는 만족하지만 배터리 지속 시간과 앱 연결 안정성에는 아쉬움이 있습니다. 연결 문제가 개선되면 추천 의향이 있습니다.","keywords":["빠른 배송","깔끔한 포장","제품 디자인","화면","배터리 소모","앱 연결 끊김","연결 안정성","추천 의향"]}

[3. JSON 파싱 성공]
{'positive': ['배송이 하루 만에 도착해 매우 빨랐음', '포장이 깔끔함', '제품 디자인과 화면이 마음에 듦'], 'negative': ['배터리가 생각보다 빨리 닳음', '앱 연결이 가끔 끊김', '연결 안정성 개선이 필요함'], 'summary': '배송과 포장, 디자인과 화면에는 만족하지만 배터리 지속 시간과 앱 연결 안정성에는 아쉬움이 있습니다. 연결 문제가 개선되면 추천 의향이 있습니다.', 'keywords': ['빠른 배송', '깔끔한 포장', '제품 디자인', '화면', '배터리 소모', '앱 연결 끊김', '연결 안정성', '추천 의향']}


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #5f6368; background: #f1f3f4; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #5f6368; letter-spacing: .02em;">🔎  선택 실험</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">동일 질문을 모델별로 비교하기</h2>
<p style="margin: 7px 0;">같은 질문으로 모델을 비교해 보세요. 응답 시간, 정확성, 지시 준수, 출력 형식을 함께 확인합니다.</p>
  </div>
</div>


In [8]:
models_to_compare = [settings.llm_model]
# AI Core에 RUNNING 상태로 배포된 모델이 있다면 아래처럼 추가합니다.
models_to_compare.append('gpt-4.1')

comparison_question = 'SAP에서 2026년도에 출시한 제품은?'
for model_name in models_to_compare:
    started_at = perf_counter()
    outcome = run_chat(comparison_question, settings, model=model_name)
    elapsed = perf_counter() - started_at
    print(f'[{model_name}] {elapsed:.1f}초')
    print(outcome['reply'])
    print('='*100 + '\n')

[gpt-5.6-luna] 9.0초
SAP는 일반 소비재처럼 매년 신제품을 한 번에 발표하기보다, 기존 클라우드 제품을 분기별·반기별로 업데이트합니다. 따라서 **“2026년에 처음 출시된 SAP 제품 전체 목록”이라는 공식 단일 목록은 없습니다.**

대표적인 SAP 제품의 최초 출시 시점은 다음과 같습니다.

| 제품 | 최초 출시·공개 시기 |
|---|---:|
| SAP S/4HANA | 2015년 |
| SAP S/4HANA Cloud | 2017년 |
| SAP Business Technology Platform(BTP) | 2021년 |
| SAP SuccessFactors | 2012년 |
| SAP Ariba | 2012년 |
| SAP Concur | 2014년 인수 후 SAP 제품군 편입 |
| SAP Business Data Cloud | 2025년 |
| SAP Joule | 2023년 |

따라서 **2026년에 나온 것은 완전히 새로운 제품보다는 SAP S/4HANA Cloud, BTP, SuccessFactors, SAP Business Data Cloud, Joule 등의 2026년 릴리스·기능 업데이트일 가능성이 큽니다.**

정확한 2026년 출시 제품이나 버전을 확인하려면 다음 SAP 공식 페이지에서 **2026년 날짜로 검색**하는 것이 가장 정확합니다.

- SAP News Center: https://news.sap.com/
- SAP Help Portal의 **Product Availability Matrix(PAM)**
- SAP 제품 릴리스 일정 및 What’s New 문서

“2026년에 출시한 제품”이 **신규 제품**, **2026년 버전**, 또는 **한국 출시 제품** 중 어느 의미인지에 따라 목록이 달라집니다.

[gpt-4.1] 2.6초
현재까지의 정보(2024년 6월 기준)로는 SAP에서 2026년도에 출시 예정이거나 출시된 제품에 대한 공식 발표가 없습니다. SAP는 주로 새로운 클라우드 솔루션,

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #7b1fa2; background: #f8f1fb; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #7b1fa2; letter-spacing: .02em;">🤖  ReAct 코드 실습</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">ReAct Agent의 답변 생성 과정</h2>
<p style="margin: 7px 0;">도구 호출부터 최종 답변까지의 흐름을 확인합니다. <code>Thinking</code>에는 다음 행동을 위한 짧은 근거만 표시합니다.</p>
  </div>
</div>


In [9]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from gen_ai_hub.proxy.native.openai import OpenAI
from src.prompt_model.lab_core import _calculator
from src.prompt_model.lesson_support.ai_core import apply_ai_core_environment

react_system_prompt = '''
Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}
'''.strip()

@tool
def calculate(expression: str) -> str:
    """사칙연산과 괄호가 있는 수식을 계산합니다."""
    return str(_calculator(expression))

apply_ai_core_environment(settings)
model = ChatOpenAI(model=settings.llm_model, api_key="EMPTY", client=OpenAI().chat.completions)
react_agent = create_agent(model=model, tools=[calculate], system_prompt=react_system_prompt)

react_question = '정가 12,000원인 상품을 15% 할인해 3개 구매하면 총 얼마인가요?'
max_react_loops = 5  # 최대 도구 실행 횟수; 답이 완성되면 더 일찍 종료
result = react_agent.invoke(
    {"messages": [{"role": "user", "content": react_question}]},
    {"recursion_limit": max_react_loops * 2 + 2},
)

for message in result["messages"]:
    if message.type == "ai" and message.content:
        print(message.content)


Question: 정가 12,000원인 상품을 15% 할인해 3개 구매하면 총 얼마인가요?  
Thought: 할인된 개당 가격을 계산한 뒤 3개를 곱하겠습니다.  
Action: calculate  
Action Input: {"expression":"12000*(1-0.15)*3"}
15% 할인된 가격은 개당 10,200원입니다.

3개 구매 시 총액은 **30,600원**입니다.


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #7b1fa2; background: #f8f1fb; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #7b1fa2; letter-spacing: .02em;">🧪 Reasoning Tag 실습</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">ReAct 실행 기록을 태그 흐름으로 읽기</h2>
<p style="margin: 7px 0;">질문을 바꿔 실행해 보세요. <code>&lt;thinking&gt;</code>은 다음 행동을 위한 짧은 근거입니다.</p>
<p style="margin: 7px 0;"><code>&lt;thinking&gt;</code>, <code>&lt;evaluation&gt;</code>, <code>&lt;answer&gt;</code>가 어떻게 이어지는지 확인합니다.</p>
  </div>
</div>


In [10]:
import json

from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from gen_ai_hub.proxy.native.openai import OpenAI

from src.prompt_model.lab_core import _calculator
from src.prompt_model.lesson_support.ai_core import apply_ai_core_environment


react_system_prompt = '''
너는 Joule 구매·회계·인사·제조원가 지원 ReAct Agent다.
Plan → Thought → Action → Observation → Evaluation → Final Answer 순서를 따른다.

절대로 Question:, Thought:, Action:, Action Input:, Observation:, Final Answer:
같은 텍스트 형식을 사용하지 않는다. XML 태그 형식만 사용한다.

태그 역할:
- <plan>: 해결할 작업의 짧은 계획
- <assumptions>: 답변에 사용한 전제
- <thinking>: 다음 행동을 선택하는 공개 가능한 짧은 근거
- <tool_selection>: 사용할 도구와 선택 이유
- <evaluation>: 도구 결과의 정확성·충분성 점검
- <reflection>: 오류·누락·추가 확인 필요 여부
- <answer>: 최종 사용자 답변

상품, 재고, 예산, 회계 정책, 직원·결재선, 제조원가, 생산능력,
외부 견적, 품질 요건은 반드시 해당 도구로 확인한다.
계산은 calculate 도구로 검증하고, 도구 반환값만 사실로 사용한다.

첫 도구 호출 전에는 다음 태그를 작성한다.

<plan>...</plan>
<assumptions>...</assumptions>
<thinking>...</thinking>
<tool_selection>...</tool_selection>

도구 결과를 받은 뒤 최종 응답은 다음 태그를 작성한다.

<thinking>...</thinking>
<evaluation>...</evaluation>
<reflection>...</reflection>
<answer>...</answer>

각 태그는 한두 문장으로 간결하게 작성한다.
상세 내부 추론이나 장문의 풀이 과정은 작성하지 않는다.
'''.strip()

@tool
def get_facility_staffing_request(site: str) -> str:
    """설비팀: 설비 운영 현황과 필요한 인력 충원 요청을 조회합니다."""
    requests = {
        "PLANT-SEOUL": {
            "request_id": "FAC-REQ-2026-041",
            "site": "PLANT-SEOUL",
            "equipment_area": "조립 라인 설비보전",
            "current_headcount": 4,
            "required_headcount": 6,
            "requested_headcount": 2,
            "position": "설비보전 Technician",
            "reason": "예방정비와 긴급 고장 대응 인력이 부족합니다.",
            "required_start_date": "2026-12-01",
        }
    }
    return json.dumps(
        requests.get(site, {"error": "설비팀 요청을 찾을 수 없습니다."}),
        ensure_ascii=False,
    )


@tool
def prepare_hr_recruitment_request(
    facility_request_id: str,
    position: str,
    headcount: int,
    required_start_date: str,
) -> str:
    """인사팀: 설비팀 인력 요청을 바탕으로 채용요청 초안을 작성합니다."""
    return json.dumps({
        "status": "DRAFT",
        "hr_request_id": "HR-REQ-2026-118",
        "facility_request_id": facility_request_id,
        "position": position,
        "requested_headcount": headcount,
        "required_start_date": required_start_date,
        "estimated_recruiting_days": 30,
        "approval_required": ["설비팀장", "HR Business Partner"],
    }, ensure_ascii=False)


@tool
def get_manufacturing_plan_impact(hr_request_id: str) -> str:
    """제조기획: 채용 요청이 생산계획과 인력 투입 일정에 미치는 영향을 확인합니다."""
    return json.dumps({
        "hr_request_id": hr_request_id,
        "production_plan": "2027년 1분기 조립라인 생산량 20% 증산",
        "recommended_onboarding_date": "2026-12-01",
        "staffing_priority": "HIGH",
        "temporary_workaround": "채용 완료 전까지 외주 정비 인력 1명 투입",
    }, ensure_ascii=False)


@tool
def get_hr_recruitment_sop(request_type: str) -> str:
    """인사팀의 채용 요청 SOP와 필수 검토 항목을 조회합니다."""
    sops = {
        "facility_staffing": {
            "sop_name": "설비 인력 충원 및 채용 요청 SOP",
            "process_sequence": [
                "설비팀 인력 충원 요청 확인",
                "인사팀 채용요청 초안 작성",
                "제조기획팀 생산계획 영향 검토",
                "설비팀장 및 HR Business Partner 결재",
            ],
            "required_fields": [
                "직무",
                "요청 인원",
                "충원 사유",
                "입사 필요일",
                "생산계획 영향",
            ],
            "restriction": "채용 공고 등록 전에는 승인된 채용요청이 필요합니다.",
        }
    }
    return json.dumps(
        sops.get(request_type, {"error": "해당 채용 SOP를 찾을 수 없습니다."}),
        ensure_ascii=False,
    )
    
react_agent = create_agent(
    model=model,
    tools=[
        get_hr_recruitment_sop,
        get_facility_staffing_request,
        prepare_hr_recruitment_request,
        get_manufacturing_plan_impact,
    ],
    system_prompt=react_system_prompt,
)

react_question = '''
PLANT-SEOUL 설비팀에서 설비보전 Technician 인력 충원이 필요합니다.

사내 인사 채용 SOP를 확인해 요청을 검토하고,
채용요청 초안을 준비한 뒤 생산계획 영향까지 확인해 주세요.
실제 채용 공고는 등록하지 않습니다.
'''

max_react_loops = 8

for update in react_agent.stream(
    {"messages": [{"role": "user", "content": react_question}]},
    {"recursion_limit": max_react_loops * 2 + 2},
    stream_mode="updates",
):
    for node_name, state in update.items():
        for message in state.get("messages", []):
            if message.type == "ai" and message.content:
                print(f"\n[{node_name}]")
                print(message.content)

            if message.type == "ai" and message.tool_calls:
                for call in message.tool_calls:
                    print(f"<action>{call['name']}({call['args']})</action>")

            if message.type == "tool":
                print(f"<observation>{message.content}</observation>")


[model]
<plan>채용 SOP와 PLANT-SEOUL 설비팀의 인력 요청을 확인한 뒤, 요청 초안을 작성하고 생산계획 영향을 검토하겠습니다. 실제 채용 공고 등록은 수행하지 않습니다.</plan>
<assumptions>사용자가 지정한 직무는 설비보전 Technician이며, 인원·입사일은 설비팀 요청과 SOP 검토 결과를 기준으로 확정합니다.</assumptions>
<thinking>SOP와 설비팀 요청은 서로 독립적으로 조회할 수 있으므로 먼저 병렬 확인하겠습니다.</thinking>
<tool_selection>get_hr_recruitment_sop는 필수 검토 항목 확인에, get_facility_staffing_request는 PLANT-SEOUL의 충원 근거·인원·희망일 확인에 사용합니다.</tool_selection>

 to=multi_tool_use.parallel.entities.functions.parallel  code:
{"tool_uses":[{"recipient_name":"functions.get_hr_recruitment_sop","parameters":{"request_type":"설비보전 Technician 신규 채용 요청"}},{"recipient_name":"functions.get_facility_staffing_request","parameters":{"site":"PLANT-SEOUL"}}]}
<thinking>두 조회 결과를 확인해 채용요청 초안에 필요한 직무·인원·입사일과 필수 검토 조건을 대조하겠습니다.</thinking>
<evaluation>도구 결과가 아직 표시되지 않아 사실값을 확인할 수 없습니다.</evaluation>
<reflection>결과가 반환되면 다음 단계로 진행하고, 반환되지 않으면 동일 조회를 재시도해야 합니다.</reflection>
<tool_selection>병렬 호출 결과가 표시되지 않았으므로, 동일한 두 항목을 개별 재조회해 정확한 반환값을 확보하겠습니다.</tool_